In [ ]:
import pickle
from collections import defaultdict

import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Load precomputed data (no model, no raw images needed) ---
with open("newt_data.pkl", "rb") as f:
    data = pickle.load(f)

filenames = data["filenames"]
labels = data["labels"]
dates = data["dates"]
embeddings = data["embeddings"]
thumbnails = data["thumbnails"]  # JPEG bytes, same order as filenames

sim_matrix = embeddings @ embeddings.T
unique_ids = sorted(set(labels))
TOP_K = 10

# --- Find top-K most similar newts (best single-image match) ---
def top_matches_for_newt(query_id, k=TOP_K):
    query_idx = np.where(labels == query_id)[0]
    other_idx = np.where(labels != query_id)[0]

    sub_sim = sim_matrix[np.ix_(query_idx, other_idx)]
    best_per_other_img = sub_sim.max(axis=0)

    scores_by_newt, best_idx_by_newt = defaultdict(float), {}
    for score, idx in zip(best_per_other_img, other_idx):
        newt_id = labels[idx]
        if score > scores_by_newt[newt_id]:
            scores_by_newt[newt_id] = score
            best_idx_by_newt[newt_id] = idx

    ranked = sorted(scores_by_newt.items(), key=lambda x: -x[1])[:k]
    return [(newt_id, score, best_idx_by_newt[newt_id]) for newt_id, score in ranked]

# --- Dropdown menu (label includes survey date) ---
dropdown_options = [(f"{nid} ({dates[np.where(labels == nid)[0][0]]})", nid) for nid in unique_ids]
dropdown = widgets.Dropdown(options=dropdown_options, description="Newt:")
output = widgets.Output()

def on_change(change):
    with output:
        clear_output(wait=True)
        query_id = change["new"]
        query_idx = np.where(labels == query_id)[0][0]

        print(f"Query: {query_id}  (date={dates[query_idx]})")
        display(widgets.Image(value=thumbnails[query_idx], format="jpeg",
                               layout=widgets.Layout(width="200px", height="200px")))

        print(f"\nTop {TOP_K} most similar newts:")
        matches = top_matches_for_newt(query_id)

        n_cols = 5
        chunks = [matches[i:i + n_cols] for i in range(0, len(matches), n_cols)]

        grid_rows = []
        for chunk in chunks:
            row_items = []
            for newt_id, score, idx in chunk:
                img_widget = widgets.Image(value=thumbnails[idx], format="jpeg",
                                            layout=widgets.Layout(width="150px", height="150px"))
                label_widget = widgets.VBox([
                    widgets.Label(f"{newt_id} ({score:.3f})"),
                    widgets.Label(f"{dates[idx]}")
                ])
                row_items.append(widgets.VBox([img_widget, label_widget]))
            grid_rows.append(widgets.HBox(row_items))

        display(widgets.VBox(grid_rows))

dropdown.observe(on_change, names="value")
display(dropdown, output)
on_change({"new": unique_ids[0]})